<a href="https://colab.research.google.com/github/mundundan-star/Citi-Bike-JC-Demand-Forecasting/blob/main/1.0%20Bronze_Data_Ingestion.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### 1.0 INTRODUCTION
This notebook downloads the Jersey City `Citi Bike` trip data zip files, extracts the csv files in memory, and uploads the parquet files to the `bronze_tripdata` folder in an  AWS S3 bucket. The Parquet format is preferred because of its `faster querying speed`, `reduced storage costs` (they compress far better than csv), and `data format preservation` (unlike csv that store everything as plain test).

In [4]:
#!pip install s3fs
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sqlalchemy import create_engine, inspect, text
from google.colab import userdata
import os

bucket_name = '2026-citibike-tripdata'

os.environ['AWS_ACCESS_KEY_ID'] = userdata.get('AWS_ACCESS_KEY_ID')
os.environ['AWS_SECRET_ACCESS_KEY'] = userdata.get('AWS_SECRET_ACCESS_KEY')
os.environ['AWS_DEFAULT_REGION'] = 'eu-west-2'

The script below automates the download of Citi Bike monthly trips data Zip files from January 2016 to July 2026, with a total of 122 Zips files downloaded using loop iterations and csvs extracted,  amounting to a total of  `6,376,426  rows` loaded, with a total size of `1,284.45 MB`.

In [ ]:
import requests, zipfile, io
from pathlib import Path
import pandas as pd

BASE_URL = "https://s3.amazonaws.com/tripdata"  # <-- confirm from a real link
START, END = (2016, 1), (2026, 8)
Path("raw_zips").mkdir(exist_ok=True)

filename = []
rows = []
y, m = START
while (y, m) <= END:
    fname = f"JC-{y}{m:02d}-citibike-tripdata.csv.zip"
    r = requests.get(f"{BASE_URL}/{fname}")
    if r.status_code != 200:
        #print("missing", fname)
        m = m % 12 + 1; y += m == 1
        continue

    (Path("raw_zips") / fname).write_bytes(r.content)

    z = zipfile.ZipFile(io.BytesIO(r.content))
    csv_name = next(n for n in z.namelist() if n.endswith(".csv"))
    data = z.read(csv_name)

    # Transforming file names from .csv to .parquet
    parquet_names = csv_name.split('.')[0] + '.parquet'
    #parquet_url = f's3://2026-citibike-tripdata/bronze_tripdata/{parquet_name}'

    # Reading csv into data
    #dataframe = pd.read_csv(z.open(csv_name))

    # Uploading files to s3 bucket as parquet files
    #dataframe.to_parquet(parquet_url, index = False)
    filenames.append(parquet_name)

    rows.append({
        "csv_file": csv_name,
        "size_mb": round(len(data) / 1_000_000, 2),
        "rows": len(pd.read_csv(io.BytesIO(data))),
    })
    #print("downloaded", fname)

    m = m % 12 + 1
    y += m == 1

summary = pd.DataFrame(rows)
#print(summary)
summary.to_csv("csv_size_summary.csv", index=False)

print(f"\nTotal rows loaded: {summary['rows'].sum():,}")
print(f"Total size loaded: {summary['size_mb'].sum():.2f} MB")

print('\nData Preview of File Uploaded to S3)
summary.head()


Total rows loaded: 6,376,426
Total size loaded: 1284.45 MB


,csv_file,size_mb,rows
0,JC-20161-citibike-tripdata.csv,1.21,7479
1,JC-20162-citibike-tripdata.csv,1.34,8250
2,JC-20163-citibike-tripdata.csv,2.18,13511
3,JC-201604-citibike-tripdata.csv,2.63,16342
4,JC-201605-citibike-tripdata.csv,3.14,19488
